# Vault with identities and creds

`.env` holds only the link to a vault. Secrets are named `<platform>/<key>` and a
plugin says which it needs. Here KeePassXC, in a temporary file with a made-up
password, so nothing else is needed.

In [ ]:
import os, tempfile
from pathlib import Path

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault), an inbox and a dataset in a temporary directory.
scratch = Path(tempfile.mkdtemp())
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = str(scratch / "inbox.sqlite3")
os.environ["AMOORION_DATASET"] = str(scratch / "dataset")

In [ ]:
from pykeepass import PyKeePass

from amoorion import Amoorion, Plugins
from amoorion.plugins import load_plugins
from amoorion.vault import VaultLink
from amoorion.vault.kpxc_db_file import initialize

file = scratch / "identity.kdbx"
link = VaultLink(kind="keepassxc", path=str(file), token="made-up")
initialize(link, {"ntfy": load_plugins().get("ntfy").schema()})

That is what `task vault:new` does. Each entry's notes say what its fields are:

In [ ]:
entry = PyKeePass(file, password="made-up").find_entries(title="ntfy", first=True)
print(entry.notes)

An identity is one env file pointing at one vault. Whether the secrets are there
shows as a flag, and values never:

In [ ]:
env_file = scratch / "identity.env"
env_file.write_text(f"AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH={file}\nAMOORION_VAULT_TOKEN=made-up\n")
amo = Amoorion(env_file)


def ntfy_ready():
    return next(status["ready"] for status in amo.accounts() if status["platform"] == "ntfy")


print(ntfy_ready(), Plugins["ntfy"].has_secrets(amo.vault))

In [ ]:
db = PyKeePass(file, password="made-up")
db.find_entries(title="ntfy", first=True).set_custom_property("topic", "my-topic")
db.save()
amo = Amoorion(env_file)
print(ntfy_ready(), Plugins["ntfy"].has_secrets(amo.vault))

`Amoorion("work.env")` and `Amoorion("private.env")` can live side by side:
`examples/advanced-usecases/scoping-of-identities-handling/`.